This code was heavily based on Enge et al. 2021 codes available at https://osf.io/34ry2/ (we are highly grateful for their commitment to open research)

In [21]:
import random
from glob import glob
from math import sqrt
from os import makedirs, path
from re import sub
from shutil import copy

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from nibabel import save
from nilearn import image, plotting, reporting
from nilearn.datasets import load_mni152_gm_template
from nilearn.image import math_img
from nimare import correct, io, meta, utils
from scipy import ndimage
from scipy.stats import norm


In [23]:
# Generate a dataset with k additional noise studies (file-drawer proxy).
def generate_null(
    text_file="peaks.txt",
    space="ale_2mm",
    k_null=100,
    random_seed=None,
    output_dir="./",
):
    # Use nilearn's MNI152 gray matter probability map, thresholded at 0.3
    gm = load_mni152_gm_template(resolution=2)
    gm_mask = math_img("img > 0.3", img=gm)

    x, y, z = np.where(gm_mask.get_fdata() == 1.0)
    within_mni = image.coord_transform(x=x, y=y, z=z, affine=gm_mask.affine)
    within_mni = np.array(within_mni).transpose()

    dset = io.convert_sleuth_to_dataset(text_file, target=space)

    if random_seed is not None:
        random.seed(random_seed)

    nr_subjects_dset = [n[0] for n in dset.metadata["sample_sizes"]]
    nr_subjects_null = random.choices(nr_subjects_dset, k=k_null)

    nr_peaks_dset = dset.coordinates["study_id"].value_counts().tolist()
    nr_peaks_null = random.choices(nr_peaks_dset, k=k_null)

    idx_list = [
        random.sample(range(len(within_mni)), k=k_peaks) for k_peaks in nr_peaks_null
    ]
    peaks_null = [within_mni[idx] for idx in idx_list]

    makedirs(output_dir, exist_ok=True)
    text_file_basename = path.basename(text_file)
    null_file_basename = sub(
        pattern=r"\.txt", repl="_plus_k" + str(k_null) + ".txt", string=text_file_basename
    )
    null_file = path.join(output_dir, null_file_basename)
    copy(text_file, null_file)

    with open(null_file, mode="a", encoding="utf-8") as f:
        for i in range(k_null):
            f.write(
                "\n// nullstudy"
                + str(i + 1)
                + "\n// Subjects="
                + str(nr_subjects_null[i])
                + "\n"
            )
            np.savetxt(f, peaks_null[i], fmt="%.3f", delimiter="\t")

    dset_null = io.convert_sleuth_to_dataset(null_file, target=space)
    return dset_null


In [24]:
# Compute cluster-level FSN using the same cluster definition as 1_ALE.ipynb.
# Original and re-estimated ALE maps are thresholded with the same
# voxel-forming and cluster-level settings before testing whether a cluster
# remains significant after adding null studies.
def compute_fsn(
    text_file="peaks.txt",
    space="ale_2mm",
    voxel_thresh=0.001,
    cluster_thresh=0.05,
    n_iters=1000,
    k_max_factor=5,
    random_ale_seed=None,
    random_null_seed=None,
    output_dir="./",
):
    print(
        "\nCOMPUTING CLUSTER-LEVEL FSN FOR "
        + text_file
        + " (seed: "
        + str(random_null_seed)
        + ")"
    )

    if random_ale_seed is not None:
        np.random.seed(random_ale_seed)

    ale = meta.cbma.ALE()
    corr = correct.FWECorrector(
        method="montecarlo", voxel_thresh=voxel_thresh, n_iters=n_iters
    )
    dset_orig = io.convert_sleuth_to_dataset(text_file=text_file, target=space)
    res_orig = ale.fit(dset_orig)
    cres_orig = corr.transform(res_orig)

    img_sig_orig_raw = cres_orig.get_map("z_desc-size_level-cluster_corr-FWE_method-montecarlo")
    cluster_thresh_z = norm.ppf(1 - cluster_thresh / 2)
    img_sig_orig = image.threshold_img(img_sig_orig_raw, threshold=cluster_thresh_z)
    sig_orig = (img_sig_orig.get_fdata() > 0).astype(np.uint8)
    img_z_orig = image.math_img(
        "img1 * img2",
        img1=image.math_img("np.where(img > 0, 1, 0)", img=img_sig_orig),
        img2=cres_orig.get_map("z"),
    )

    prefix = path.basename(text_file).replace(".txt", "")
    makedirs(output_dir, exist_ok=True)

    if not np.any(sig_orig):
        empty = pd.DataFrame(
            columns=[
                "Cluster ID", "Peak X", "Peak Y", "Peak Z",
                "Peak Z-stat", "Cluster Voxels", "Cluster Size (mm3)", "FSN",
            ]
        )
        empty.to_csv(path.join(output_dir, prefix + "_fsn.tsv"), sep="\t", index=False)
        img_empty = image.new_img_like(img_sig_orig, np.zeros_like(sig_orig, dtype=float))
        save(img_empty, filename=path.join(output_dir, prefix + "_fsn.nii.gz"))
        return img_empty, empty

    structure = np.ones((3, 3, 3), dtype=np.uint8)
    labels_orig, n_clusters = ndimage.label(sig_orig, structure=structure)

    ids_orig = dset_orig.ids.tolist()
    k_max = len(ids_orig) * k_max_factor
    dset_null = generate_null(
        text_file=text_file,
        space=space,
        k_null=k_max,
        random_seed=random_null_seed,
        output_dir=output_dir,
    )

    # FSN per cluster: starts at 0, increments only while cluster stays significant
    fsn_per_cluster = np.zeros(n_clusters, dtype=int)

    for k in range(1, k_max + 1):
        print("  Computing ALE with k=" + str(k) + " null studies...")

        ids_null = ["nullstudy" + str(x) + "-" for x in range(1, k + 1)]
        dset_k = dset_null.slice(ids_orig + ids_null)
        ale_k = meta.cbma.ALE()
        corr_k = correct.FWECorrector(
            method="montecarlo", voxel_thresh=voxel_thresh, n_iters=n_iters
        )
        res_k = ale_k.fit(dset_k)
        cres_k = corr_k.transform(result=res_k)
        img_k_raw = cres_k.get_map("z_desc-size_level-cluster_corr-FWE_method-montecarlo")
        img_k = image.threshold_img(img_k_raw, threshold=cluster_thresh_z)
        sig_k = img_k.get_fdata() > 0

        any_updated = False
        for cid in range(1, n_clusters + 1):
            cluster_mask = labels_orig == cid
            # Only increment if cluster was continuously significant up to k-1
            if fsn_per_cluster[cid - 1] == k - 1 and np.any(sig_k & cluster_mask):
                fsn_per_cluster[cid - 1] = k
                any_updated = True

        # Early stopping: no cluster remained significant at this k
        if not any_updated:
            print("  No cluster updated at k=" + str(k) + " — stopping early.")
            break

    for cid in range(1, n_clusters + 1):
        print("  Cluster " + str(cid) + ": FSN = " + str(fsn_per_cluster[cid - 1]))

    # Build FSN map
    fsn_map = np.zeros_like(sig_orig, dtype=float)
    for cid in range(1, n_clusters + 1):
        fsn_map[labels_orig == cid] = fsn_per_cluster[cid - 1]

    img_fsn = image.new_img_like(img_sig_orig, fsn_map)
    save(img_fsn, filename=path.join(output_dir, prefix + "_fsn.nii.gz"))

    z_data = img_z_orig.get_fdata()
    voxel_vol = abs(np.linalg.det(img_sig_orig.affine[:3, :3]))

    rows = []
    for cid in range(1, n_clusters + 1):
        cmask = labels_orig == cid
        voxels = int(cmask.sum())
        idx = np.argwhere(cmask)
        peak_idx = idx[np.argmax(z_data[cmask])]
        peak_val = float(z_data[tuple(peak_idx)])
        peak_x, peak_y, peak_z = image.coord_transform(
            x=float(peak_idx[0]),
            y=float(peak_idx[1]),
            z=float(peak_idx[2]),
            affine=img_sig_orig.affine,
        )
        rows.append({
            "Cluster ID": cid,
            "Peak X": float(peak_x),
            "Peak Y": float(peak_y),
            "Peak Z": float(peak_z),
            "Peak Z-stat": peak_val,
            "Cluster Voxels": voxels,
            "Cluster Size (mm3)": float(voxels * voxel_vol),
            "FSN": int(fsn_per_cluster[cid - 1]),
        })

    tab_fsn = pd.DataFrame(rows)
    tab_fsn.to_csv(path.join(output_dir, prefix + "_fsn.tsv"), sep="\t", index=False)

    return img_fsn, tab_fsn


In [26]:
# Runtime configuration (edit this cell before running)
PREFIXES = ["hyper_and_hypo"]
NR_FILEDRAWERS = 5
VOXEL_THRESH = 0.001
CLUSTER_THRESH = 0.05
N_ITERS = 10000
K_MAX_FACTOR = 5
RANDOM_ALE_SEED = 2025

# Absolute project root
PROJECT_DIR = "/Volumes/ss/Self_Psych_Meta"

# Build input/output paths from prefixes
text_files = [f"{PROJECT_DIR}/1_Data/AnalysisData/InputData_ALE/{prefix}.txt" for prefix in PREFIXES]
output_dirs = [f"{PROJECT_DIR}/3_Output/7_FSN/{prefix}/" for prefix in PREFIXES]


In [27]:
# Random seeds for filedrawers
random_null_seeds = random.sample(range(1000), k=NR_FILEDRAWERS)
filedrawers = ["filedrawer" + str(seed) for seed in random_null_seeds]


In [ ]:
def _run_one_filedrawer(text_file, output_dir, filedrawer, random_null_seed):
    return compute_fsn(
        text_file=text_file,
        space="ale_2mm",
        voxel_thresh=VOXEL_THRESH,
        cluster_thresh=CLUSTER_THRESH,
        n_iters=N_ITERS,
        k_max_factor=K_MAX_FACTOR,
        random_ale_seed=RANDOM_ALE_SEED,
        random_null_seed=random_null_seed,
        output_dir=output_dir + filedrawer,
    )


# Run filedrawers (serial execution to avoid CPU contention)
for text_file, output_dir in zip(text_files, output_dirs):
    for filedrawer, seed in zip(filedrawers, random_null_seeds):
        _run_one_filedrawer(text_file, output_dir, filedrawer, seed)


# Aggregate across filedrawers from THIS run's filedrawer list only
for prefix in PREFIXES:
    fnames_maps = [f"{PROJECT_DIR}/3_Output/7_FSN/{prefix}/{fd}/{prefix}_fsn.nii.gz" for fd in filedrawers]
    fnames_maps = [f for f in fnames_maps if path.exists(f)]
    imgs_fsn = [image.load_img(fname) for fname in fnames_maps]
    imgs_fsn = [
        image.new_img_like(img, img.get_fdata().astype(np.float32), img.affine)
        for img in imgs_fsn
    ]

    img_mean = image.mean_img(imgs_fsn)
    fname_img_mean = f"{PROJECT_DIR}/3_Output/7_FSN/{prefix}/{prefix}_mean_fsn.nii.gz"
    save(img_mean, fname_img_mean)

    fnames_tabs = [f"{PROJECT_DIR}/3_Output/7_FSN/{prefix}/{fd}/{prefix}_fsn.tsv" for fd in filedrawers]
    fnames_tabs = [f for f in fnames_tabs if path.exists(f)]
    tabs_fsn = [pd.read_csv(fname, delimiter="\t") for fname in fnames_tabs]
    tabs_fsn = [t for t in tabs_fsn if not t.empty]

    fname_agg = f"{PROJECT_DIR}/3_Output/7_FSN/{prefix}/{prefix}_mean_fsn.csv"
    if len(tabs_fsn) == 0:
        empty = pd.DataFrame(
            columns=["Cluster ID", "mean", "count", "std", "se", "ci_lower", "ci_upper"]
        )
        empty.to_csv(fname_agg, index=False)
        continue

    tab_fsn = pd.concat(tabs_fsn)
    agg = tab_fsn.groupby("Cluster ID")["FSN"].agg(["mean", "count", "std"])

    z_crit = abs(norm.ppf(0.05 / 2))
    agg["se"] = [
        (std / sqrt(count)) if (count > 0 and pd.notna(std)) else 0
        for std, count in zip(agg["std"], agg["count"])
    ]
    agg["ci_lower"] = agg["mean"] - z_crit * agg["se"]
    agg["ci_upper"] = agg["mean"] + z_crit * agg["se"]
    agg.to_csv(fname_agg, float_format="%.3f")



COMPUTING CLUSTER-LEVEL FSN FOR /Volumes/ss/Self_Psych_Meta/1_Data/AnalysisData/InputData_ALE/hyper_and_hypo.txt (seed: 982)


INFO:nimare.correct:Using correction method implemented in Estimator: nimare.meta.cbma.ale.ALE.correct_fwe_montecarlo.
100%|██████████| 10000/10000 [1:00:00<00:00,  2.78it/s]
INFO:nimare.meta.cbma.base:Using null distribution for voxel-level FWE correction.
/var/folders/yk/78rqxlxn4pz_rsb5_31xvh340000gn/T/ipykernel_51075/784768105.py:37: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  img_sig_orig = image.threshold_img(img_sig_orig_raw, threshold=cluster_thresh_z)
/Users/ss/miniconda3/lib/python3.12/site-packages/nilearn/image/image.py:1160: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for 

  Computing ALE with k=1 null studies...


INFO:nimare.correct:Using correction method implemented in Estimator: nimare.meta.cbma.ale.ALE.correct_fwe_montecarlo.
100%|██████████| 10000/10000 [55:25<00:00,  3.01it/s] 
INFO:nimare.meta.cbma.base:Using null distribution for voxel-level FWE correction.
/var/folders/yk/78rqxlxn4pz_rsb5_31xvh340000gn/T/ipykernel_51075/784768105.py:88: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  img_k = image.threshold_img(img_k_raw, threshold=cluster_thresh_z)


  Computing ALE with k=2 null studies...


...

INFO:nimare.correct:Using correction method implemented in Estimator: nimare.meta.cbma.ale.ALE.correct_fwe_montecarlo.
100%|██████████| 10000/10000 [52:38<00:00,  3.17it/s] 
INFO:nimare.meta.cbma.base:Using null distribution for voxel-level FWE correction.
/var/folders/yk/78rqxlxn4pz_rsb5_31xvh340000gn/T/ipykernel_51075/784768105.py:88: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  img_k = image.threshold_img(img_k_raw, threshold=cluster_thresh_z)


  Computing ALE with k=5 null studies...


INFO:nimare.correct:Using correction method implemented in Estimator: nimare.meta.cbma.ale.ALE.correct_fwe_montecarlo.
100%|██████████| 10000/10000 [57:02<00:00,  2.92it/s] 
INFO:nimare.meta.cbma.base:Using null distribution for voxel-level FWE correction.
/var/folders/yk/78rqxlxn4pz_rsb5_31xvh340000gn/T/ipykernel_51075/784768105.py:88: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  img_k = image.threshold_img(img_k_raw, threshold=cluster_thresh_z)


  Computing ALE with k=6 null studies...


INFO:nimare.correct:Using correction method implemented in Estimator: nimare.meta.cbma.ale.ALE.correct_fwe_montecarlo.
 69%|██████▉   | 6929/10000 [40:11<25:16,  2.03it/s]  